In [1]:
import sys
import asyncio
import time
import os
import matplotlib.pyplot as plt
import numpy as np
import yaml
from lsst.ts import salobj
from lsst.ts.externalscripts.auxtel.latiss_cwfs_align import LatissCWFSAlign
from lsst.ts.idl.enums.Script import ScriptState

/home/craiglagegit/WORK/ts_externalscripts/python/lsst/ts/externalscripts/auxtel/latiss_acquire_and_take_sequence.py:37: UserWarning: Cannot import lsst.observing. Script will not work.
  warnings.warn("Cannot import lsst.observing. Script will not work.")
/home/craiglagegit/WORK/ts_externalscripts/python/lsst/ts/externalscripts/auxtel/latiss_acquire_target.py:41: UserWarning: Cannot import lsst.observing. Script will not work.
  warnings.warn("Cannot import lsst.observing. Script will not work.")
/home/craiglagegit/WORK/ts_externalscripts/python/lsst/ts/externalscripts/auxtel/latiss_take_sequence.py:36: UserWarning: Cannot import lsst.observing. Script will not work.
  warnings.warn("Cannot import lsst.observing. Script will not work.")


In [2]:
import logging
stream_handler = logging.StreamHandler(sys.stdout)
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG

In [3]:
# turn off logging for matplotlib
mpl_logger = logging.getLogger('matplotlib')
mpl_logger.setLevel(logging.WARNING)

In [4]:
cwfs = LatissCWFSAlign(index=1, remotes=True)

atmcs: Adding all resources.
atptg: Adding all resources.
ataos: Adding all resources.
atpneumatics: Adding all resources.
athexapod: Adding all resources.
atdome: Adding all resources.
atdometrajectory: Adding all resources.
atcamera: Adding all resources.
atspectrograph: Adding all resources.
atheaderservice: Adding all resources.
atarchiver: Adding all resources.
Read historical data in 0.11 sec
Read 1 history items for RemoteEvent(ATArchiver, 0, authList)
Read 100 history items for RemoteEvent(ATArchiver, 0, heartbeat)
Read 100 history items for RemoteEvent(ATArchiver, 0, imageInOODS)
Read 100 history items for RemoteEvent(ATArchiver, 0, imageRetrievalForArchiving)
Read 1 history items for RemoteEvent(ATArchiver, 0, logLevel)
Read 1 history items for RemoteEvent(ATArchiver, 0, logMessage)
Read 1 history items for RemoteEvent(ATArchiver, 0, simulationMode)
Read 1 history items for RemoteEvent(ATArchiver, 0, softwareVersions)
Read historical data in 0.16 sec
Read 1 history items for 

In [5]:
await cwfs.start_task

In [7]:
await cwfs.atcs.slew_object('HIP 20733')

Starting new HTTP connection (1): simbad.u-strasbg.fr:80
http://simbad.u-strasbg.fr:80 "POST /simbad/sim-script HTTP/1.1" 200 None
Slewing to HIP 20733: 04 26 36.5752 -55 12 35.198
Auto sky angle: 0.0 deg
Sending command
Stop tracking.
target python read queue is filling: 51 of 100 elements
Unknown tracking state: 9.
Unknown tracking state: 10.
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
atdome: <State.ENABLED: 2>
atdometrajectory: <State.ENABLED: 2>
[Telescope] delta Alt = +008.169 deg; delta Az = -001.572 deg; delta N1 = +000.000 deg; delta N2 = +010.524 deg [Dome] delta Az = +000.232 deg
ATDome in position.
[Telescope] delta Alt = +006.552 deg; delta Az = -000.466 deg; delta N1 = +000.000 deg; delta N2 = +009.262 deg [Dome] delta Az = +000.232 deg
[Telescope] delta Alt = +002.807 deg; delta Az = +000.001 deg; delta N1

In [8]:
await cwfs.latiss.take_object(20.0, 1, filter='empty_1', grating='empty_1')

Generating group_id
OBJECT 0001 - 0001


array([2021012100703])

In [9]:
cwfs.atcs.rem.atptg.cmd_raDecTarget.set(azWrapStrategy=1)  # 1 does not unwrap, 0 unwraps

In [10]:
configuration = yaml.safe_dump({"filter": 'empty_1', 
                                "grating": 'empty_1',
                                "exposure_time": 20,
                                "dataPath": '/project/shared/auxTel/rerun/quickLook'})
print(configuration)

dataPath: /project/shared/auxTel/rerun/quickLook
exposure_time: 20
filter: empty_1
grating: empty_1



In [11]:
config_data = cwfs.cmd_configure.DataType()
config_data.config = configuration
await cwfs.do_configure(config_data)

Using binning factor of 2


In [12]:
cwfs.set_state(ScriptState.UNCONFIGURED)

In [13]:
cwfs.intra_visit_id = None
cwfs.extra_visit_id = None
cwfs.short_timeout = 10

In [14]:
await cwfs.run_cwfs()

Intra/Extra images not taken. Running take image sequence.
Moving to intra-focal position
ENGTEST 0001 - 0001
Moving to extra-focal position
Taking extra-focal image
ENGTEST 0001 - 0001
logMessage DDS read queue is filling: 53 of 100 elements
Moving hexapod back to zero offset (in-focus) position
intraImage expId for target: 2021012100704
extraImage expId for target: 2021012100705
Checking for header correction file named LATISS-AT_O_20210121_000704.yaml
AT_O_20210121_000704: Forcing detector serial to ITL-3800C-068
AT_O_20210121_000704: Forcing SHUTTIME header to be None
Checking for header correction file named LATISS-AT_O_20210121_000704.yaml
AT_O_20210121_000704: Forcing detector serial to ITL-3800C-068
AT_O_20210121_000704: Forcing SHUTTIME header to be None
Checking for header correction file named LATISS-AT_O_20210121_000704.yaml
AT_O_20210121_000704: Forcing detector serial to ITL-3800C-068
AT_O_20210121_000704: Forcing SHUTTIME header to be None
Checking for header correction 

RuntimeError: Unable to retrieve bias for {'expId': 2021012100704, 'dayObs': '2021-01-21', 'detector': 0}: No registry for lookup.

logMessage DDS read queue is filling: 82 of 100 elements
target DDS read queue is filling: 97 of 100 elements
target python read queue is filling: 96 of 100 elements
logMessage DDS read queue is filling: 11 of 100 elements
target DDS read queue is filling: 16 of 100 elements
target python read queue is filling: 15 of 100 elements
target DDS read queue is filling: 86 of 100 elements
target python read queue is filling: 85 of 100 elements
target python read queue is filling: 15 of 100 elements
target DDS read queue is filling: 43 of 100 elements
target python read queue is filling: 42 of 100 elements
target python read queue is filling: 46 of 100 elements


In [ ]:
await cwfs.latiss.take_object(20.0, 1, filter='empty_1', grating='empty_1')

In [ ]:
fig1 = plt.figure(1, figsize=(12,8))
ax11 = fig1.add_subplot(121)
ax11.set_title(f"intra - {cwfs.intra_visit_id}")
ax11.imshow(cwfs.I1[0].image0)
ax11.contour(cwfs.algo.pMask) 
ax12 = fig1.add_subplot(122)
ax12.set_title(f"extra - {cwfs.extra_visit_id}")
ax12.imshow(cwfs.I2[0].image0)
ax12.contour(cwfs.algo.pMask) 

In [ ]:
await cwfs.atcs.rem.ataos.cmd_disableCorrection.set_start(atspectrograph=True)

In [ ]:
await cwfs.latiss.take_object(20.0, 1, filter='RG610', grating='empty_1')

In [ ]:
configuration = yaml.safe_dump({"filter": 'RG610', 
                                "grating": 'empty_1',
                                "exposure_time": 20,
                                "dataPath": '/project/shared/auxTel/rerun/quickLook'})
print(configuration)

In [ ]:
config_data = cwfs.cmd_configure.DataType()
config_data.config = configuration
await cwfs.do_configure(config_data)

In [ ]:
cwfs.intra_visit_id = None
cwfs.extra_visit_id = None
cwfs.short_timeout = 10

In [ ]:
await cwfs.run_cwfs()

In [ ]:
fig1 = plt.figure(1, figsize=(12,8))
ax11 = fig1.add_subplot(121)
ax11.set_title(f"intra - {cwfs.intra_visit_id}")
ax11.imshow(cwfs.I1[0].image0)
ax11.contour(cwfs.algo.pMask) 
ax12 = fig1.add_subplot(122)
ax12.set_title(f"extra - {cwfs.extra_visit_id}")
ax12.imshow(cwfs.I2[0].image0)
ax12.contour(cwfs.algo.pMask) 

In [ ]:
await cwfs.atcs.rem.ataos.cmd_offset.set_start(z=-0.02949709)

In [ ]:
cwfs.intra_visit_id = None
cwfs.extra_visit_id = None
cwfs.short_timeout = 10

In [ ]:
await cwfs.run_cwfs()

In [ ]:
await cwfs.latiss.take_object(20.0, 1, filter='RG610', grating='empty_1')

In [ ]:
test = await cwfs.atcs.rem.ataos.evt_focusOffsetSummary.aget()
print(test)

In [ ]:
await cwfs.latiss.setup_atspec(filter='BG40', grating='empty_1')

In [ ]:
configuration = yaml.safe_dump({"filter": 'BG40', 
                                "grating": 'empty_1',
                                "exposure_time": 20,
                                "dataPath": '/project/shared/auxTel/rerun/quickLook'})
print(configuration)

In [ ]:
cwfs.set_state(ScriptState.UNCONFIGURED)

In [ ]:
config_data = cwfs.cmd_configure.DataType()
config_data.config = configuration
await cwfs.do_configure(config_data)

In [ ]:
cwfs.intra_visit_id = None
cwfs.extra_visit_id = None
cwfs.short_timeout = 10

In [ ]:
await cwfs.run_cwfs()

In [ ]:
fig1 = plt.figure(1, figsize=(12,8))
ax11 = fig1.add_subplot(121)
ax11.set_title(f"intra - {cwfs.intra_visit_id}")
ax11.imshow(cwfs.I1[0].image0)
ax11.contour(cwfs.algo.pMask) 
ax12 = fig1.add_subplot(122)
ax12.set_title(f"extra - {cwfs.extra_visit_id}")
ax12.imshow(cwfs.I2[0].image0)
ax12.contour(cwfs.algo.pMask) 

In [ ]:
await cwfs.latiss.take_object(20.0, 1, filter='BG40', grating='empty_1')

In [ ]:
# Now running a focus sweep with the grating in place 

In [ ]:
await cwfs.latiss.take_object(20.0, 1, filter='empty_1', grating='ronchi90lpmm')

In [ ]:
test = await cwfs.atcs.rem.ataos.evt_focusOffsetSummary.aget()
print(test)

In [ ]:
import math
#Focus units are given in mm 
focus_center = -0.2155
focus_step = 0.02
n_steps = 9
focus_positions = np.linspace(focus_center - focus_step * math.floor(n_steps / 2), focus_center + focus_step * math.ceil(n_steps / 2 - 1), n_steps)
#focus_positions = [0.05, -0.05, -0.100, -0.150, -0.200, -0.250, -0.300, -0.3, -0.2, -0.2, -0.24, -0.26, -0.28]
print ('focus_positions = ' + str([pos for pos in focus_positions]))

In [ ]:
exp_time = 5.0 
filter_str = 'empty_1'
grating_str = 'ronchi90lpmm'

In [ ]:
await cwfs.atcs.slew_object('HD61597', rot_type=2)

In [ ]:
await cwfs.latiss.take_object(20.0, 1, filter='empty_1', grating='ronchi90lpmm')

In [ ]:
test = await cwfs.atcs.rem.ataos.evt_focusOffsetSummary.aget()
print(test)

In [ ]:
#!!!!DO NOT RUN UNLESS YOU HAVE BEEN CLEARED TO CONTROL TELESCOPE!!!!! 
focus_images = [0 for focus_pos in focus_positions]
for i in range(len(focus_positions)): 
    focus_pos = focus_positions[i] 
    print ('Working on focus position ' + str(focus_pos))
    await cwfs.atcs.focus_offset(focus_pos)
    focus_image = await cwfs.latiss.take_engtest(exptime=exp_time, n=1, filter=filter_str, grating=grating_str) 
    print ('Newest image id is = ' + str(focus_image ) )
    focus_images[i] = int(focus_image )
#This resets the focus position after taking this sequence: 
await cwfs.atcs.focus_offset(focus_pos)

In [ ]:
await cwfs.latiss.take_object(20.0, 1, filter='empty_1', grating='ronchi90lpmm')

In [ ]:
test = await cwfs.atcs.rem.ataos.evt_focusOffsetSummary.aget()
print(test)

In [ ]:
await cwfs.atcs.focus_offset(0.2955)

In [ ]:
await cwfs.atcs.rem.ataos.cmd_resetOffset.set_start(axis='z')

In [ ]:
test = await cwfs.atcs.rem.ataos.evt_focusOffsetSummary.aget()
print(test)

In [ ]:
await cwfs.atcs.focus_offset(-0.0740367397665977)

In [ ]:
test = await cwfs.atcs.rem.ataos.evt_focusOffsetSummary.aget()
print(test)

In [ ]:
await cwfs.latiss.take_object(20.0, 1, filter='empty_1', grating='ronchi90lpmm')

In [ ]:
#Focus units are given in mm 
focus_center = 0.0
focus_step = 0.02
n_steps = 9
focus_positions = np.linspace(focus_center - focus_step * math.floor(n_steps / 2), focus_center + focus_step * math.ceil(n_steps / 2 - 1), n_steps)
#focus_positions = [0.05, -0.05, -0.100, -0.150, -0.200, -0.250, -0.300, -0.3, -0.2, -0.2, -0.24, -0.26, -0.28]
print ('focus_positions = ' + str([pos for pos in focus_positions]))

In [ ]:
#!!!!DO NOT RUN UNLESS YOU HAVE BEEN CLEARED TO CONTROL TELESCOPE!!!!! 
focus_images = [0 for focus_pos in focus_positions]
for i in range(len(focus_positions)): 
    focus_pos = focus_positions[i] 
    print ('Working on focus position ' + str(focus_pos))
    await cwfs.atcs.focus_offset(focus_pos)
    focus_image = await cwfs.latiss.take_engtest(exptime=exp_time, n=1, filter=filter_str, grating=grating_str) 
    print ('Newest image id is = ' + str(focus_image ) )
    focus_images[i] = int(focus_image )
#This resets the focus position after taking this sequence: 
await cwfs.atcs.focus_offset(focus_pos)

In [ ]:
await cwfs.atcs.rem.ataos.cmd_resetOffset.set_start(axis='z')

In [ ]:
test = await cwfs.atcs.rem.ataos.evt_focusOffsetSummary.aget()
print(test)

In [ ]:
await cwfs.atcs.focus_offset(-0.0740367397665977)

In [ ]:
test = await cwfs.atcs.rem.ataos.evt_focusOffsetSummary.aget()
print(test)

In [ ]:
focus_window = 0.1
n_step = 9
focus_step = 0.1 / n_step
print(focus_step)

In [ ]:
await cwfs.atcs.focus_offset(-focus_window/2.)
for i in range(len(focus_positions)): 
    await cwfs.atcs.focus_offset(focus_step)
    focus_image = await cwfs.latiss.take_engtest(exptime=exp_time, n=1, filter=filter_str, grating=grating_str) 
    print ('Newest image id is = ' + str(focus_image ) )
#     focus_images[i] = int(focus_image )
#This resets the focus position after taking this sequence: 
await cwfs.atcs.focus_offset(-focus_window/2.)

In [ ]:
focus_window = 0.13
focus_center = 0.03
n_step = 9
focus_step = 0.13 / n_step
print(focus_step)
exp_time = 30.0

In [ ]:
await cwfs.atcs.focus_offset(focus_center)

In [ ]:
await cwfs.atcs.focus_offset(-focus_window/2.)
for i in range(n_step): 
    await cwfs.atcs.focus_offset(focus_step)
    focus_image = await cwfs.latiss.take_engtest(exptime=exp_time, n=1, filter=filter_str, grating=grating_str) 
    print ('Newest image id is = ' + str(focus_image ) )
#     focus_images[i] = int(focus_image )
#This resets the focus position after taking this sequence: 
await cwfs.atcs.focus_offset(-focus_window/2.)

In [ ]:
focus_window = 0.13
n_step = 21
focus_step = focus_window / n_step
print(focus_step)
exp_time = 30.0

In [ ]:
await cwfs.atcs.focus_offset(-focus_window/2.)
for i in range(n_step+1): 
    await cwfs.atcs.focus_offset(focus_step)
    focus_image = await cwfs.latiss.take_engtest(exptime=exp_time, n=1, filter=filter_str, grating=grating_str) 
    print ('Newest image id is = ' + str(focus_image ) )
#     focus_images[i] = int(focus_image )
#This resets the focus position after taking this sequence: 
await cwfs.atcs.focus_offset(-focus_window/2.)

In [ ]:
await cwfs.latiss.take_object(30.0, 1, filter='empty_1', grating='ronchi90lpmm')

In [ ]:
await cwfs.latiss.take_object(10.0, 1, filter='empty_1', grating='ronchi90lpmm')

In [ ]:
await cwfs.latiss.take_object(10.0, 1, filter='empty_1', grating='ronchi90lpmm')

In [ ]:
await cwfs.latiss.take_object(10.0, 1, filter='empty_1', grating='ronchi90lpmm')

In [ ]:
focus_window = 0.13
n_step = 21
focus_step = focus_window / n_step
print(focus_step)
exp_time = 30.0

In [ ]:
await cwfs.atcs.focus_offset(-focus_window/2.)
for i in range(n_step+1): 
    await cwfs.atcs.focus_offset(focus_step)
    focus_image = await cwfs.latiss.take_engtest(exptime=exp_time, n=1, filter=filter_str, grating=grating_str) 
    print ('Newest image id is = ' + str(focus_image ) )
#     focus_images[i] = int(focus_image )
#This resets the focus position after taking this sequence: 
await cwfs.atcs.focus_offset(-focus_window/2.)

In [ ]:
await cwfs.atcs.slew_object('HIP72704', rot_type=2)

In [ ]:
await cwfs.latiss.take_object(10.0, 1, filter='empty_1', grating='empty_1')

In [ ]:
await cwfs.atcs.shutdown()

In [ ]:
cwfs.atcs.check.atdome=True

In [ ]:
await cwfs.latiss.standby()